# Stub per funzione Deutsch

funzioni di Qiskit di cui bisogna creare lo stub:
- `QuantumCircuit(num_qubits,num_clbits)`
- `qc.x(qubit)`
- `qc.h(qubit)`
- `qc.cx(control, target)`
- `qc.measure(qubit, clbits)`

### COME VA MODIFICATA LA FUNZIONE DEUTSCH PER ESSERE VERIFICATA DA NAGINI

* `qc.h(range(2))` va sostituita con `qc.h(0)` e `qc.h(1)`, perché in Qiskit `h` accetta interi, liste, range e altri tipi, ma Nagini non gestisce bene gli overloa -> lo stub accetta solo `qubit: int`
* il ramo `else: raise ValueError` va sostituito dalla precondizione `Requires(0 <= caso and caso <= 3)`, che evita di gestire le eccezioni con `Exsures`
* ho aggiunto `Ensures(Acc(Result().campo))` per restituire al chiamante il permesso di accesso ai campi del circuito
* ho aggiunto `Ensures` sui valori: `Result().num_qubits == 2` e `Result().num_clbits == 1`
* `Result()` indica il valore restituito dalla funzione, per questo si può usare solo nelle postcondizioni (`Ensures`)
    * in questo caso è il circuito `qc` restituito da `deutsch`
* `Implies(a, b)` vuol dire “se `a` è vero, allora vale `b`”, quindi ogni riga lega un valore di caso al numero di porte atteso.
* * il ramo `caso == 0` corrisponde all'oracolo senza porte

In [ ]:

from nagini_contracts.contracts import *
#from qiskit_stub import QuantumCircuit

def deutsch(caso: int) -> QuantumCircuit:
    Requires(0 <= caso and caso <= 3)
        
    Ensures(Acc(Result().num_qubits) and Result().num_qubits == 2) # garantiamo accesso al numero di qubit e che siano 2
    Ensures(Acc(Result().num_clbits) and Result().num_clbits == 1)
    Ensures(Acc(Result().gates))
    
    Ensures(Implies(caso == 0, Result().gates == 5))   # f(x) = 0 -> numero di porte base
    Ensures(Implies(caso == 1, Result().gates == 6))   # f(x) = x
    Ensures(Implies(caso == 2, Result().gates == 7))   # f(x) = NOT x
    Ensures(Implies(caso == 3, Result().gates == 6))   # f(x) = 1
    
    qc = QuantumCircuit(2, 1)
    qc.x(1)
    #qc.h(range(2))
    qc.h(0)
    qc.h(1)
    
    if caso == 0:
        pass
    elif caso == 1:
        qc.cx(0, 1)
    elif caso == 2:
        qc.cx(0, 1)
        qc.x(1)
    elif caso == 3:
        qc.x(1)
    
    qc.h(0)
    qc.measure(0, 0)
    return qc

    

### IMPLEMENTAZIONE STUB

REQUIRES:
- esprime le condizioni che gli argomenti devono rispettare; Nagini controlla che ogni chiamata le soddisfi
- con `Acc` il metodo riceve il permesso di leggere o scrivere i campi
    * In `def x(...)` senza `Acc(self.num_qubits, ...)` non si potrebbe scrivere la condizione `qubit < self.num_qubits`, perché leggere un campo richiede il permesso -> è garantito da `Acc(self.num_qubits, 1/2)`
    *   permesso 1/2 (qualsiasi frazione minore di 1) -> solo lettura
    *   permesso 1 -> lettura e scrittura

ENSURES:
- In `def __init__(...)` il chiamante riceve il permesso pieno sui campi del nuovo circuito e sa con quali valori sono inizializzati
- negli altri stub
    - `Ensures(Acc(self.num_qubits, 1/2) and Acc(self.gates))` -> restituisce al chiamante i permessi ricevuti con Requires
    - `Ensures(self.gates == Old(self.gates) + 1)` -> ogni porta aggiunge esattamente un'operazione al circuito

RICORDA:
* `Acc` in `Requires`: il chiamante cede il permesso al metodo
* `Acc` in `Ensures`: il metodo restituisce il permesso al chiamante
* Se un metodo prende un permesso con `Requires` e non lo restituisce con `Ensures`, il chiamante lo perde e non può più usare quel campo

In [ ]:
#from nagini_contracts.contracts import *

class QuantumCircuit:
    def __init__(self, num_qubits: int, num_clbits: int) -> None:
        Requires(num_qubits > 0 and num_clbits >= 0)
        
        Ensures(Acc(self.num_qubits) and self.num_qubits == num_qubits)
        Ensures(Acc(self.num_clbits) and self.num_clbits == num_clbits)
        Ensures(Acc(self.gates) and self.gates == 0)    
        
        self.num_qubits: int = num_qubits
        self.num_clbits: int = num_clbits
        self.gates: int = 0

    def x(self, qubit: int) -> None:
        Requires(Acc(self.num_qubits, 1/2) and Acc(self.gates))
        Requires(0 <= qubit and qubit < self.num_qubits)
        
        Ensures(Acc(self.num_qubits, 1/2) and Acc(self.gates))
        Ensures(self.gates == Old(self.gates) + 1)
        
        self.gates = self.gates + 1

    def h(self, qubit: int) -> None:
        Requires(Acc(self.num_qubits, 1/2) and Acc(self.gates))
        Requires(0 <= qubit and qubit < self.num_qubits)
        
        Ensures(Acc(self.num_qubits, 1/2) and Acc(self.gates))
        Ensures(self.gates == Old(self.gates) + 1)
        
        self.gates = self.gates + 1

    def cx(self, control: int, target: int) -> None:
        Requires(Acc(self.num_qubits, 1/2) and Acc(self.gates))
        Requires(0 <= control and control < self.num_qubits)
        Requires(0 <= target and target < self.num_qubits)
        Requires(control != target)
        
        Ensures(Acc(self.num_qubits, 1/2) and Acc(self.gates))
        Ensures(self.gates == Old(self.gates) + 1)
        
        self.gates = self.gates + 1

    def measure(self, qubit: int, clbit: int) -> None:
        Requires(Acc(self.num_qubits, 1/2) and Acc(self.num_clbits, 1/2) and Acc(self.gates))
        Requires(0 <= qubit and qubit < self.num_qubits)
        Requires(0 <= clbit and clbit < self.num_clbits)
        
        Ensures(Acc(self.num_qubits, 1/2) and Acc(self.num_clbits, 1/2) and Acc(self.gates))
        Ensures(self.gates == Old(self.gates) + 1)
        
        self.gates = self.gates + 1

ERRORE DATO A TERMINALE.
- Type error: Cannot determine type of "num_qubits"  [has-type] (qiskit_stub.py@7.20)
- Type error: Cannot determine type of "num_clbits"  [has-type] (qiskit_stub.py@8.20)
- Type error: Cannot determine type of "gates"  [has-type] (qiskit_stub.py@9.20)

MOTIVAZIONE: Nei contratti del costruttore ho usato self.num_qubits, self.num_clbits e self.gates prima di assegnarli
- Nagini usa mypy per ricavare i tipi, e mypy non riesce a dedurre il tipo di un campo che compare prima della sua assegnazione.

SOLUZIONE: dichiarare il tipo dei campi nel costruttore, self.num_qubits: int = num_qubits, così mypy conosce il tipo anche nei contratti scritti prima dell’assegnazione

RISULTATO FINALE: 
Verification successful
Verification took 8.64 seconds.